# Cellule 1 : Installation des dépendances

In [5]:
!pip install coqui-tts
!pip install datasets
!pip install soundfile
!pip install librosa

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.7 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=8066fbe5c0aaa10b2e0e6f0a51f2fa57ec9c8e560f74bbaa0f34d2efbc82ab5f
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


# Cellule 2 : Imports et configuration

In [6]:
import os
import torch
import soundfile as sf
import librosa
import numpy as np
from datasets import load_dataset
from TTS.api import TTS
from IPython.display import Audio
import matplotlib.pyplot as plt

# Cellule 3 : Chargement du dataset

In [26]:
from datasets import load_dataset, Audio

def load_malagasy_tts_dataset():
    """Charge le dataset audio malgache"""
    print("Chargement du dataset audio malgache...")
    dataset = load_dataset(path='hasiniaina/malagasy-female-speech-dataset')

    # Vérifier la structure brute
    print(f"\nStructure du dataset:")
    print(f"   Features: {dataset['train'].column_names}")
    print(f"   Nombre d'échantillons: {len(dataset['train'])}")

    # Examiner le premier élément sans charger l'audio
    first = dataset['train'][0]
    print("\nPremier élément (sans audio chargé):")
    for k, v in first.items():
        if k == 'wav':
            print(f"   wav: {v} (type: {type(v)})")
        else:
            print(f"   {k}: {v[:50] if isinstance(v, str) else v}")

    # Charger les audios avec cast_column
    print("\nChargement des fichiers audio...")
    dataset = dataset.cast_column('wav', Audio(sampling_rate=22050))

    # Vérifier un exemple avec audio chargé
    sample = dataset['train'][0]
    print(f"\nExemple après chargement:")
    print(f"   Texte: {sample['text_norm'][:80]}...")
    print(f"   Audio array shape: {sample['wav']['array'].shape}")
    print(f"   Sampling rate: {sample['wav']['sampling_rate']}")
    print(f"   Durée: {len(sample['wav']['array']) / sample['wav']['sampling_rate']:.2f}s")

    return dataset

dataset = load_malagasy_tts_dataset()

Chargement du dataset audio malgache...

Structure du dataset:
   Features: ['wav', 'text_norm']
   Nombre d'échantillons: 2226

Premier élément (sans audio chargé):
   wav: <datasets.features._torchcodec.AudioDecoder object at 0x7b344d403290> (type: <class 'datasets.features._torchcodec.AudioDecoder'>)
   text_norm: andian-javatra roa lehibe no maha-firenena ny fire

Chargement des fichiers audio...

Exemple après chargement:
   Texte: andian-javatra roa lehibe no maha-firenena ny firenena....
   Audio array shape: (85655,)
   Sampling rate: 22050
   Durée: 3.88s


Cellule 4 : Analyse exploratoire du dataset

In [27]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

def analyze_dataset(dataset):
    """
    Analyse statistique du dataset avec gestion d'erreurs.
    """
    # Charger les fichiers audio à la demande via la colonne 'wav'
    # On récupère le nombre d'échantillons et le sample rate
    print("Analyse des fichiers audio...")

    durations = []
    texts = []
    errors = 0

    # Pour éviter de tout charger en mémoire, on itère par index
    for idx in tqdm(range(len(dataset['train']))):
        try:
            sample = dataset['train'][idx]
            # Obtenir l'audio et le sampling rate via 'wav' (après cast_column)
            wav_dict = sample['wav']
            array = wav_dict['array']
            sr = wav_dict['sampling_rate']
            duration = len(array) / sr
            durations.append(duration)
            texts.append(sample['text_norm'])
        except Exception as e:
            errors += 1
            if errors <= 5:  # Afficher les premières erreurs seulement
                print(f"Erreur à l'index {idx}: {e}")
            continue

    text_lengths = [len(t) for t in texts]

    print("\nAnalyse statistique:")
    print(f"   Échantillons analysés: {len(durations)}")
    print(f"   Échecs: {errors}")
    print(f"   Durée moyenne: {np.mean(durations):.2f}s")
    print(f"   Durée min/max: {min(durations):.2f}s / {max(durations):.2f}s")
    print(f"   Longueur texte moyenne: {np.mean(text_lengths):.1f} caractères")
    print(f"   Longueur texte min/max: {min(text_lengths)} / {max(text_lengths)}")

    # Histogramme des durées
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.hist(durations, bins=20, color='steelblue', edgecolor='black')
    plt.xlabel('Durée (secondes)')
    plt.ylabel('Nombre')
    plt.title('Distribution des durées audio')

    plt.subplot(1, 2, 2)
    plt.hist(text_lengths, bins=20, color='coral', edgecolor='black')
    plt.xlabel('Longueur du texte (caractères)')
    plt.ylabel('Nombre')
    plt.title('Distribution des longueurs de texte')

    plt.tight_layout()
    plt.show()

    return durations, text_lengths

# Exécuter
durations, text_lengths = analyze_dataset(dataset)

Analyse des fichiers audio...


 93%|█████████▎| 2073/2226 [00:31<00:03, 44.37it/s]

Erreur à l'index 2067: Tried to read outside of the buffer: current_pos=4271878338, size=111166
Erreur à l'index 2068: Tried to read outside of the buffer: current_pos=25364048, size=328966
Erreur à l'index 2070: Tried to read outside of the buffer: current_pos=96480262, size=337678
Erreur à l'index 2071: Tried to read outside of the buffer: current_pos=13625222, size=190300
Erreur à l'index 2072: Tried to read outside of the buffer: current_pos=7282476, size=138754


100%|██████████| 2226/2226 [00:33<00:00, 66.79it/s]



Analyse statistique:
   Échantillons analysés: 2089
   Échecs: 137
   Durée moyenne: 5.22s
   Durée min/max: 0.99s / 13.30s
   Longueur texte moyenne: 75.8 caractères
   Longueur texte min/max: 12 / 180


# Cellule 5 : Écouter un échantillon audio

In [28]:
def listen_sample(dataset, index=0):
    """Écouter un échantillon audio du dataset"""
    sample = dataset['train'][index]
    audio_array = sample['wav']['array']
    sample_rate = sample['wav']['sampling_rate']
    text = sample['text_norm']

    print(f"Texte: {text}")
    print(f"Durée: {len(audio_array)/sample_rate:.2f}s")

    # Lecture dans Colab
    from IPython.display import Audio
    display(Audio(audio_array, rate=sample_rate))

    # Affichage de la forme d'onde
    plt.figure(figsize=(12, 3))
    time = np.linspace(0, len(audio_array)/sample_rate, len(audio_array))
    plt.plot(time, audio_array)
    plt.xlabel('Temps (s)')
    plt.ylabel('Amplitude')
    plt.title('Forme d\'onde audio')
    plt.show()

    return audio_array, sample_rate

# Écouter le premier échantillon
audio, sr = listen_sample(dataset, 0)

Texte: andian-javatra roa lehibe no maha-firenena ny firenena.
Durée: 3.88s


# Cellule 6 : Approche 1 - Fine-tuning avec Coqui TTS

In [13]:
import soundfile as sf
import os
from tqdm import tqdm

def setup_finetuning(dataset, output_dir="./malagasy_tts"):
    """
    Prépare les données pour le fine-tuning avec Coqui TTS.
    Ignore les fichiers audio corrompus.
    """
    os.makedirs(output_dir, exist_ok=True)

    audio_files = []
    text_files = []
    errors = 0

    for idx in tqdm(range(len(dataset['train']))):
        try:
            sample = dataset['train'][idx]
            audio_array = sample['wav']['array']
            sample_rate = sample['wav']['sampling_rate']
            text = sample['text_norm']

            # Sauvegarder l'audio
            audio_path = f"{output_dir}/audio_{idx:04d}.wav"
            sf.write(audio_path, audio_array, sample_rate)

            # Sauvegarder la transcription
            text_path = f"{output_dir}/text_{idx:04d}.txt"
            with open(text_path, 'w', encoding='utf-8') as f:
                f.write(text)

            audio_files.append(audio_path)
            text_files.append(text_path)

        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"Erreur à l'index {idx}: {e}")
            continue

    # Créer le fichier metadata
    metadata_path = f"{output_dir}/metadata.csv"
    with open(metadata_path, 'w', encoding='utf-8') as f:
        for i, (audio_path, text_path) in enumerate(zip(audio_files, text_files)):
            audio_filename = os.path.basename(audio_path)
            with open(text_path, 'r', encoding='utf-8') as tf:
                text = tf.read().strip()
            f.write(f"{audio_filename}|{text}\n")

    print(f"\nDonnées préparées dans {output_dir}")
    print(f"   Fichiers audio valides: {len(audio_files)} / {len(dataset['train'])}")
    print(f"   Échecs: {errors}")
    print(f"   Fichier metadata: {metadata_path}")

    return metadata_path

metadata_path = setup_finetuning(dataset)

 93%|█████████▎| 2079/2226 [00:47<00:03, 46.63it/s]

Erreur à l'index 2067: Tried to read outside of the buffer: current_pos=4271878338, size=111166
Erreur à l'index 2068: Tried to read outside of the buffer: current_pos=25364048, size=328966
Erreur à l'index 2070: Tried to read outside of the buffer: current_pos=96480262, size=337678
Erreur à l'index 2071: Tried to read outside of the buffer: current_pos=13625222, size=190300
Erreur à l'index 2072: Tried to read outside of the buffer: current_pos=7282476, size=138754


100%|██████████| 2226/2226 [00:49<00:00, 45.10it/s]



✅ Données préparées dans ./malagasy_tts
   Fichiers audio valides: 2089 / 2226
   Échecs: 137
   Fichier metadata: ./malagasy_tts/metadata.csv


# Cellule 7 : Sauvegarde du modèle et des métadonnées pour réutilisation

In [25]:
import json
import shutil

def save_model_artifacts(output_dir="./malagasy_tts", model_dir="./malagasy_model"):
    """
    Sauvegarde les artefacts nécessaires pour réutiliser la voix sans recharger tout le dataset.
    """
    os.makedirs(model_dir, exist_ok=True)

    # 1. Copier la voix de référence (exemple)
    ref_wav = None
    for f in os.listdir(output_dir):
        if f.startswith("audio_") and f.endswith(".wav"):
            ref_wav = os.path.join(output_dir, f)
            break
    if ref_wav:
        shutil.copy(ref_wav, os.path.join(model_dir, "reference_voice.wav"))
        print(f"Voix de référence copiée: {ref_wav}")

    # 2. Sauvegarder les métadonnées
    metadata = {
        "num_samples": len([f for f in os.listdir(output_dir) if f.startswith("audio_")]),
        "language": "mg",
        "model_type": "XTTS-v2",
        "reference_voice": "reference_voice.wav"
    }
    with open(os.path.join(model_dir, "model_info.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Artefacts sauvegardés dans {model_dir}")

# Exécuter
save_model_artifacts()

Voix de référence copiée: ./malagasy_tts/audio_0192.wav
Artefacts sauvegardés dans ./malagasy_model


# Cellule 8 : Fonctions prêtes pour l'intégration web (API Flask / FastAPI)

In [24]:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf
from IPython.display import Audio

class MalagasyTTSWrapper:
    """
    Wrapper pour le modèle MMS-TTS de Facebook, spécialisé pour le malgache.
    """
    def __init__(self):
        print("Initialisation du TTS malgache avec MMS-TTS...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"   Utilisation du device: {self.device}")

        # Charger le modèle MMS-TTS pour le malgache (code langue: mlg)
        self.model = VitsModel.from_pretrained("facebook/mms-tts-mlg").to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-mlg")

        print("Modèle prêt")

    def synthesize(self, text, output_path=None):
        """
        Synthétise la parole à partir du texte en malgache.
        """
        if not text:
            raise ValueError("Le texte est vide")

        if output_path is None:
            output_path = "output_tts.wav"

        # Tokenisation
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        # Génération audio
        with torch.no_grad():
            output = self.model(**inputs).waveform

        waveform = output.squeeze().cpu().numpy()
        sf.write(output_path, waveform, self.model.config.sampling_rate)

        # Lecture dans le notebook
        display(Audio(waveform, rate=self.model.config.sampling_rate))

        return output_path

# Exemple d'utilisation
tts_wrapper = MalagasyTTSWrapper()
test_output = tts_wrapper.synthesize("Misaotra betsaka amin'ny fampiasana ity fitaovana ity.")
print(f"Fichier généré: {test_output}")

Initialisation du TTS malgache avec MMS-TTS...
   Utilisation du device: cpu


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Modèle prêt


Fichier généré: output_tts.wav


# Cellule 9 : Démonstration finale et export pour éditeur web

In [ ]:
!pip install gtts
!pip install pydub

In [ ]:
import base64

def export_for_web():
    """
    Génère un fichier audio exemple en base64 pour intégration dans le frontend.
    """
    # Créer quelques exemples de phrases
    phrases = [
        "Manao ahoana ny zanakao?",
        "Misaotra betsaka",
        "Veloma",
        "Tonga soa eto"
    ]

    output_dir = "web_samples"
    os.makedirs(output_dir, exist_ok=True)

    for i, phrase in enumerate(phrases):
        audio_path = tts_wrapper.synthesize(phrase, output_path=f"{output_dir}/sample_{i}.wav")

        # Convertir en base64 pour affichage direct dans le frontend
        with open(audio_path, "rb") as f:
            audio_base64 = base64.b64encode(f.read()).decode("utf-8")

        print(f"Phrase {i+1}: {phrase}")
        print(f"   Audio (base64): {audio_base64[:100]}...")
        print(f"   Fichier: {audio_path}")
        print()

    # Créer un fichier JSON avec tous les exemples
    manifest = {
        "samples": [
            {"text": phrase, "audio_base64": base64.b64encode(open(f"{output_dir}/sample_{i}.wav", "rb").read()).decode()}
            for i, phrase in enumerate(phrases)
        ]
    }
    with open(f"{output_dir}/manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print(f" Exports terminés dans {output_dir}")
    print(" Contient: 4 fichiers .wav + manifest.json")

export_for_web()

In [22]:
from transformers import VitsModel, AutoTokenizer

# Télécharger depuis Hugging Face
model = VitsModel.from_pretrained("facebook/mms-tts-mlg")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-mlg")

# Sauvegarder dans un dossier local (par exemple "./mms_tts_malagasy")
model.save_pretrained("./mms_tts_malagasy")
tokenizer.save_pretrained("./mms_tts_malagasy")

print("Modèle sauvegardé dans ./mms_tts_malagasy")

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle sauvegardé dans ./mms_tts_malagasy
